# Traceability — ledger ↔ artifacts ↔ exhibits

Every number below is read from an aggregate result artifact in `../artifacts/` — the same files the paper's
tables and figures were generated from. No licensed microdata is used or required (see `../DATA_ACCESS.md`).


## Every claim in the ledger resolves to its artifact
`CLAIMS_LEDGER_v4.csv` lists each reported number with the artifact file and JSON path it comes from.
This cell follows every path and compares the stored value with the artifact. Rows whose path points at a
whole object (a sum or a range computed from several fields) are reported as *derived*. The one artifact
deliberately excluded from the repository (`I05.json`, see `ARTIFACT_MANIFEST.md`) is reported as missing.


In [ ]:
import json, os, csv
ART = "../artifacts"
rows = list(csv.DictReader(open(os.path.join(ART, "CLAIMS_LEDGER_v4.csv"), encoding="utf-8-sig")))
def resolve(o, path):
    for k in [k for k in path.split(".") if k]:
        o = o[int(k)] if isinstance(o, list) else o[k]
    return o
exact = derived = mismatch = missing = 0; bad = []
for r in rows:
    f = os.path.join(ART, os.path.basename(r["source_json"]))
    if not os.path.exists(f):
        missing += 1; bad.append((r["claim_id"], "artifact not in repository", os.path.basename(r["source_json"]))); continue
    try: o = resolve(json.load(open(f, encoding="utf-8")), r["json_path"])
    except Exception: mismatch += 1; bad.append((r["claim_id"], "path", r["json_path"])); continue
    if isinstance(o, (dict, list)): derived += 1; continue
    try: ok = abs(float(r["value"]) - float(o)) <= max(5e-5, abs(float(o)) * 1e-6)
    except Exception: ok = str(o) == r["value"]
    if ok: exact += 1
    else: mismatch += 1; bad.append((r["claim_id"], "value", f"{r['value']} vs {o}"))
print(f"ledger rows {len(rows)} · exact {exact} · derived {derived} · mismatch {mismatch} · artifact missing {missing}")
for b in bad: print("  ", b)
from collections import Counter
print("\nrows by status:", dict(Counter(r["path_status"] for r in rows)))
print("SUPERSEDED rows point at retired designs; they are kept so that every number ever reported stays traceable.")
assert mismatch == 0


ledger rows 350 · exact 344 · derived 6 · mismatch 0 · artifact missing 0

rows by status: {'GO': 233, 'PARTIAL': 63, 'KILL': 13, 'SUPERSEDED': 19, 'OK': 22}
SUPERSEDED rows point at retired designs; they are kept so that every number ever reported stays traceable.


## Artifact integrity
Recomputes SHA-256 for every artifact and compares with `../ARTIFACT_MANIFEST.md`.


In [ ]:
import hashlib, re
man = open("../ARTIFACT_MANIFEST.md", encoding="utf-8").read()
listed = dict(re.findall(r"^\| `([^`]+\.json)` \| `([0-9a-f]{16})` \|", man, re.M))
bad = [f for f, h in listed.items() if hashlib.sha256(open(os.path.join(ART, f), "rb").read()).hexdigest()[:16] != h]
print(f"artifacts listed {len(listed)} · hash mismatches {len(bad)}"); assert not bad
extra = sorted(set(f for f in os.listdir(ART) if f.endswith(".json")) - set(listed)); print("unlisted artifacts:", extra); assert not extra


artifacts listed 64 · hash mismatches 0
unlisted artifacts: []


## Which artifacts feed which exhibits


In [ ]:
import re
use = {}
for nb, lab in [("01_main_tables.ipynb", "tables"), ("02_figures.ipynb", "figures")]:
    src = "\n".join("".join(c["source"]) for c in json.load(open(nb, encoding="utf-8"))["cells"] if c["cell_type"] == "code")
    for a in sorted(set(re.findall(r'J\("([A-Za-z0-9_]+)"\)', src))): use.setdefault(a, set()).add(lab)
led = {}
for r in rows:
    k = os.path.basename(r["source_json"]).replace(".json", ""); led[k] = led.get(k, 0) + 1
print(f"{'artifact':<28}{'read directly by':<20}{'ledger rows'}")
for a in sorted(set(use) | set(led)): print(f"{a:<28}{'/'.join(sorted(use.get(a, []))) or '—':<20}{led.get(a, 0)}")


artifact                    read directly by    ledger rows
I01                         figures             1
I02                         —                   1
I03                         —                   2
I04c                        —                   2
I11                         —                   1
I14                         —                   2
I15                         —                   1
I16                         —                   1
I17                         —                   2
I21                         —                   1
I22                         —                   2
I25                         —                   1
I31                         figures             2
I32                         figures             3
I33                         figures             2
I35                         figures             11
I36                         tables              0
I37                         tables              0
I38                         figures    